# Indar - GPU server (free Colab GPU)

Runs the AI room redesign (Dreamshaper 8 img2img) on Google's free GPU and gives your backend a public URL.

**One-time setup (about 1 minute)** - open the key icon (Secrets) in the left bar and add three secrets, switching "Notebook access" ON for each:

| Secret name | Value |
|---|---|
| `SUPABASE_URL` | your Supabase project URL |
| `SUPABASE_ANON_KEY` | your Supabase publishable / anon key |
| `GPU_TOKEN` | the token in `colab/gpu_token.local.txt` on your PC |

**Every time you use the app**
1. Menu **Runtime -> Change runtime type -> T4 GPU -> Save**
2. Menu **Runtime -> Run all** (first run takes about 6-8 minutes: it downloads Dreamshaper 8 and the ControlNet, about 3.5 GB)
3. Wait for `Registered with your Indar backend` at the bottom. Nothing to copy: the backend picks the new URL up by itself within about 10 seconds.
4. Keep this tab open while you use the app.

If the secrets are missing the notebook still prints a `COMFYUI_URL=...` line you can paste into `indar_app_backend/.env` by hand.

In [ ]:
# 1. Check that a GPU is attached
!nvidia-smi --query-gpu=name,memory.total --format=csv || echo "NO GPU - set Runtime > Change runtime type > T4 GPU"

In [ ]:
# 2. Install the server and its dependencies
!mkdir -p /content/indar && cd /content/indar &&   wget -q -O server.py https://raw.githubusercontent.com/Abhin611/Interior-AI/main/gpu_server/server.py &&   wget -q -O requirements.txt https://raw.githubusercontent.com/Abhin611/Interior-AI/main/gpu_server/requirements.txt
!pip install -q -r /content/indar/requirements.txt
!wget -q -O /usr/local/bin/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 && chmod +x /usr/local/bin/cloudflared
print("installed")

In [ ]:
# 3. Download Dreamshaper 8 + the ControlNet that keeps your room's shape (about 3.5 GB, cached for this session)
import sys
sys.path.insert(0, "/content/indar")
import os; os.chdir("/content/indar")
import server
server.get_pipe()
print("model ready; ControlNet on:", server._cn_ok)

In [ ]:
# 4. Start the API server in the background
import threading, time, requests, uvicorn
threading.Thread(target=lambda: uvicorn.run(server.app, host="0.0.0.0", port=8188, log_level="warning"), daemon=True).start()
for _ in range(30):
    try:
        print(requests.get("http://127.0.0.1:8188/system_stats").json()); break
    except Exception:
        time.sleep(1)

In [ ]:
# 5. Open a public tunnel and register its URL with your backend
import subprocess, re, time, requests
tunnel = subprocess.Popen(["cloudflared", "tunnel", "--url", "http://127.0.0.1:8188", "--no-autoupdate"],
                          stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
url = None
for line in tunnel.stdout:
    m = re.search(r"https://[a-z0-9-]+[.]trycloudflare[.]com", line)
    if m:
        url = m.group(0)
        break
time.sleep(3)

def secret(name):
    try:
        from google.colab import userdata
        return userdata.get(name)
    except Exception:
        return None

sb_url, sb_key, token = secret("SUPABASE_URL"), secret("SUPABASE_ANON_KEY"), secret("GPU_TOKEN")
print()
print("=" * 60)
registered = False
if sb_url and sb_key and token:
    try:
        r = requests.post(sb_url.rstrip("/") + "/rest/v1/rpc/set_gpu_url",
                          headers={"apikey": sb_key, "Authorization": "Bearer " + sb_key},
                          json={"p_token": token, "p_url": url}, timeout=30)
        registered = r.status_code == 200
        if not registered:
            print("Could not register:", r.status_code, r.text[:200])
    except Exception as e:
        print("Could not register:", e)
if registered:
    print("Registered with your Indar backend. The app can use the AI now.")
    print("GPU server URL:", url)
else:
    print("Add the Colab secrets (see the top of this notebook) to skip this step next time.")
    print("For now put this line in indar_app_backend/.env and restart the backend:")
    print()
    print(f"COMFYUI_URL={url}")
print("=" * 60)

In [ ]:
# 6. Keep the session alive (stop this cell to shut the server down)
import time
while True:
    time.sleep(60)